# Day 15 — Streamlit Setup & Interactive Prediction App

## EduPro Predictive Modeling

### Objective

Create a Streamlit application that uses the final Day 14
prediction pipelines to provide interactive:

- EnrollmentCount prediction
- CourseRevenue prediction

The application will recreate the Day 13 feature-engineering
logic required by the final Enrollment model.

In [2]:
# ============================================================
# DAY 15 — IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np

from pathlib import Path

import joblib

print("All Day 15 libraries imported successfully.")

All Day 15 libraries imported successfully.


In [3]:
# ============================================================
# DAY 15 PATH CONFIGURATION
# ============================================================

project_folder = Path(
    r"D:\Data Analytics Project\EduPro_Predictive_Modeling"
)

model_folder = project_folder / "models"
app_folder = project_folder / "app"
notebook_folder = project_folder / "notebooks"

day14_file_path = (
    project_folder
    / "data"
    / "model_data"
    / "EduPro_Day14_Final_Model_Predictions.xlsx"
)

enrollment_model_path = (
    model_folder
    / "EduPro_Final_Enrollment_Model.joblib"
)

revenue_model_path = (
    model_folder
    / "EduPro_Final_Revenue_Model.joblib"
)

app_file_path = app_folder / "app.py"

requirements_file_path = (
    app_folder / "requirements.txt"
)

app_folder.mkdir(
    parents=True,
    exist_ok=True
)

notebook_folder.mkdir(
    parents=True,
    exist_ok=True
)

print("Project folder:")
print(project_folder)

print("\nModels folder:")
print(model_folder)

print("\nDay 14 output:")
print(day14_file_path)

print("\nEnrollment model:")
print(enrollment_model_path)

print("\nRevenue model:")
print(revenue_model_path)

print("\nStreamlit app:")
print(app_file_path)

print("\nRequirements:")
print(requirements_file_path)

Project folder:
D:\Data Analytics Project\EduPro_Predictive_Modeling

Models folder:
D:\Data Analytics Project\EduPro_Predictive_Modeling\models

Day 14 output:
D:\Data Analytics Project\EduPro_Predictive_Modeling\data\model_data\EduPro_Day14_Final_Model_Predictions.xlsx

Enrollment model:
D:\Data Analytics Project\EduPro_Predictive_Modeling\models\EduPro_Final_Enrollment_Model.joblib

Revenue model:
D:\Data Analytics Project\EduPro_Predictive_Modeling\models\EduPro_Final_Revenue_Model.joblib

Streamlit app:
D:\Data Analytics Project\EduPro_Predictive_Modeling\app\app.py

Requirements:
D:\Data Analytics Project\EduPro_Predictive_Modeling\app\requirements.txt


## Validate Day 14 artifacts

In [4]:
# ============================================================
# DAY 14 ARTIFACT VALIDATION
# ============================================================

required_files = {
    "Day 14 output": day14_file_path,
    "Enrollment model": enrollment_model_path,
    "Revenue model": revenue_model_path
}

print("========== DAY 14 ARTIFACT VALIDATION ==========")

for name, path in required_files.items():

    print(f"\n{name}:")
    print(path)
    print("Exists:", path.exists())

    if not path.exists():
        raise FileNotFoundError(
            f"{name} not found:\n{path}"
        )

print("\nAll Day 14 artifacts are available.")

========== DAY 14 ARTIFACT VALIDATION ==========

Day 14 output:
D:\Data Analytics Project\EduPro_Predictive_Modeling\data\model_data\EduPro_Day14_Final_Model_Predictions.xlsx
Exists: True

Enrollment model:
D:\Data Analytics Project\EduPro_Predictive_Modeling\models\EduPro_Final_Enrollment_Model.joblib
Exists: True

Revenue model:
D:\Data Analytics Project\EduPro_Predictive_Modeling\models\EduPro_Final_Revenue_Model.joblib
Exists: True

All Day 14 artifacts are available.


## Load final model

In [5]:
# ============================================================
# LOAD FINAL DAY 14 MODELS
# ============================================================

enrollment_model = joblib.load(
    enrollment_model_path
)

revenue_model = joblib.load(
    revenue_model_path
)

print("Enrollment model loaded:")
print(type(enrollment_model))

print("\nRevenue model loaded:")
print(type(revenue_model))

Enrollment model loaded:
<class 'sklearn.pipeline.Pipeline'>

Revenue model loaded:
<class 'sklearn.pipeline.Pipeline'>


## Define original features

In [6]:
# ============================================================
# ORIGINAL MODELING FEATURES
# ============================================================

original_features = [
    "CourseCategory",
    "CourseType",
    "CourseLevel",
    "CoursePrice",
    "CourseDuration",
    "CourseRating",
    "TeacherRating",
    "YearsOfExperience",
    "Expertise"
]

target_columns = [
    "EnrollmentCount",
    "CourseRevenue"
]

print("Original features:")
print(original_features)

print("\nTargets:")
print(target_columns)

Original features:
['CourseCategory', 'CourseType', 'CourseLevel', 'CoursePrice', 'CourseDuration', 'CourseRating', 'TeacherRating', 'YearsOfExperience', 'Expertise']

Targets:
['EnrollmentCount', 'CourseRevenue']


## Define engineered features

In [7]:
# ============================================================
# DAY 13 ENGINEERED FEATURES
# ============================================================

engineered_features = [
    "PricePerDay",
    "PriceSquared",
    "DurationSquared",
    "LogCoursePrice",
    "LogCourseDuration",
    "RatingGap",
    "AverageRating",
    "CourseQualityScore",
    "ExperienceRatingScore",
    "PriceRatingInteraction",
    "PricePerRatingPoint",
    "Category_Type",
    "Category_Level",
    "Type_Level",
    "Category_Expertise",
    "Level_Expertise",
    "ExperienceBand"
]

all_modeling_features = (
    original_features +
    engineered_features
)

print(
    "Original features:",
    len(original_features)
)

print(
    "Engineered features:",
    len(engineered_features)
)

print(
    "Total modeling features:",
    len(all_modeling_features)
)

Original features: 9
Engineered features: 17
Total modeling features: 26


## Create reusable feature engineering function 

In [8]:
# ============================================================
# REUSABLE DAY 13 FEATURE ENGINEERING
# ============================================================

def create_engineered_features(df):

    data = df.copy()

    # --------------------------------------------------------
    # Price / duration features
    # --------------------------------------------------------

    data["PricePerDay"] = (
        data["CoursePrice"]
        /
        data["CourseDuration"].replace(0, np.nan)
    )

    data["PricePerDay"] = (
        data["PricePerDay"]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )

    data["PriceSquared"] = (
        data["CoursePrice"] ** 2
    )

    data["DurationSquared"] = (
        data["CourseDuration"] ** 2
    )

    data["LogCoursePrice"] = np.log1p(
        data["CoursePrice"]
    )

    data["LogCourseDuration"] = np.log1p(
        data["CourseDuration"]
    )

    # --------------------------------------------------------
    # Quality / rating features
    # --------------------------------------------------------

    data["RatingGap"] = (
        data["CourseRating"]
        -
        data["TeacherRating"]
    )

    data["AverageRating"] = (
        data["CourseRating"]
        +
        data["TeacherRating"]
    ) / 2

    data["CourseQualityScore"] = (
        data["CourseRating"]
        *
        data["TeacherRating"]
    )

    data["ExperienceRatingScore"] = (
        data["YearsOfExperience"]
        *
        data["TeacherRating"]
    )

    # --------------------------------------------------------
    # Price / quality interaction
    # --------------------------------------------------------

    data["PriceRatingInteraction"] = (
        data["CoursePrice"]
        *
        data["AverageRating"]
    )

    data["PricePerRatingPoint"] = (
        data["CoursePrice"]
        /
        data["AverageRating"].replace(0, np.nan)
    )

    data["PricePerRatingPoint"] = (
        data["PricePerRatingPoint"]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )

    # --------------------------------------------------------
    # Categorical interactions
    # --------------------------------------------------------

    data["Category_Type"] = (
        data["CourseCategory"].astype(str)
        + "_"
        + data["CourseType"].astype(str)
    )

    data["Category_Level"] = (
        data["CourseCategory"].astype(str)
        + "_"
        + data["CourseLevel"].astype(str)
    )

    data["Type_Level"] = (
        data["CourseType"].astype(str)
        + "_"
        + data["CourseLevel"].astype(str)
    )

    data["Category_Expertise"] = (
        data["CourseCategory"].astype(str)
        + "_"
        + data["Expertise"].astype(str)
    )

    data["Level_Expertise"] = (
        data["CourseLevel"].astype(str)
        + "_"
        + data["Expertise"].astype(str)
    )

    # --------------------------------------------------------
    # Experience band
    # --------------------------------------------------------

    data["ExperienceBand"] = pd.cut(
        data["YearsOfExperience"],
        bins=[
            -np.inf,
            5,
            10,
            20,
            np.inf
        ],
        labels=[
            "Early",
            "Developing",
            "Experienced",
            "Highly_Experienced"
        ]
    )

    return data


print(
    "Day 13 feature engineering function "
    "recreated successfully."
)

Day 13 feature engineering function recreated successfully.


## Test the feature-engineering function

In [9]:
# ============================================================
# TEST FEATURE ENGINEERING
# ============================================================

test_course = pd.DataFrame([
    {
        "CourseCategory": "Technology",
        "CourseType": "Online",
        "CourseLevel": "Intermediate",
        "CoursePrice": 490.9,
        "CourseDuration": 7.55,
        "CourseRating": 4.55,
        "TeacherRating": 4.58,
        "YearsOfExperience": 24,
        "Expertise": "Programming"
    }
])

test_engineered = create_engineered_features(
    test_course
)

print("Engineered test row shape:")
print(test_engineered.shape)

print("\nRequired engineered features:")
print(
    test_engineered[
        engineered_features
    ].T
)

Engineered test row shape:
(1, 26)

Required engineered features:
                                               0
PricePerDay                            65.019868
PriceSquared                           240982.81
DurationSquared                          57.0025
LogCoursePrice                          6.198275
LogCourseDuration                       2.145931
RatingGap                                  -0.03
AverageRating                              4.565
CourseQualityScore                        20.839
ExperienceRatingScore                     109.92
PriceRatingInteraction                 2240.9585
PricePerRatingPoint                   107.535597
Category_Type                  Technology_Online
Category_Level           Technology_Intermediate
Type_Level                   Online_Intermediate
Category_Expertise        Technology_Programming
Level_Expertise         Intermediate_Programming
ExperienceBand                Highly_Experienced


## Test both final models

In [10]:
# ============================================================
# TEST FINAL DAY 14 MODELS
# ============================================================

test_original = test_course[
    original_features
].copy()

test_engineered_input = test_engineered[
    all_modeling_features
].copy()

test_enrollment_prediction = (
    enrollment_model.predict(
        test_engineered_input
    )[0]
)

test_revenue_prediction = (
    revenue_model.predict(
        test_original
    )[0]
)

# Prevent negative business predictions
test_enrollment_prediction = max(
    0,
    test_enrollment_prediction
)

test_revenue_prediction = max(
    0,
    test_revenue_prediction
)

print("Test Enrollment Prediction:")
print(round(test_enrollment_prediction, 2))

print("\nTest Revenue Prediction:")
print(round(test_revenue_prediction, 2))

Test Enrollment Prediction:
168.23

Test Revenue Prediction:
82525.93


## Genrate the Streamlit application

In [11]:
# ============================================================
# CREATE STREAMLIT APPLICATION
# ============================================================

app_code = r'''
import streamlit as st
import pandas as pd
import numpy as np
import joblib
from pathlib import Path


# ============================================================
# PATHS
# ============================================================

APP_FOLDER = Path(__file__).resolve().parent
PROJECT_FOLDER = APP_FOLDER.parent
MODELS_FOLDER = PROJECT_FOLDER / "models"

ENROLLMENT_MODEL_PATH = (
    MODELS_FOLDER /
    "EduPro_Final_Enrollment_Model.joblib"
)

REVENUE_MODEL_PATH = (
    MODELS_FOLDER /
    "EduPro_Final_Revenue_Model.joblib"
)


# ============================================================
# PAGE CONFIGURATION
# ============================================================

st.set_page_config(
    page_title="EduPro Predictive Intelligence",
    page_icon="📊",
    layout="wide"
)


# ============================================================
# LOAD MODELS
# ============================================================

@st.cache_resource
def load_models():

    enrollment_model = joblib.load(
        ENROLLMENT_MODEL_PATH
    )

    revenue_model = joblib.load(
        REVENUE_MODEL_PATH
    )

    return enrollment_model, revenue_model


enrollment_model, revenue_model = load_models()


# ============================================================
# FEATURE ENGINEERING
# ============================================================

def create_engineered_features(df):

    data = df.copy()

    data["PricePerDay"] = (
        data["CoursePrice"]
        /
        data["CourseDuration"].replace(0, np.nan)
    )

    data["PricePerDay"] = (
        data["PricePerDay"]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )

    data["PriceSquared"] = (
        data["CoursePrice"] ** 2
    )

    data["DurationSquared"] = (
        data["CourseDuration"] ** 2
    )

    data["LogCoursePrice"] = np.log1p(
        data["CoursePrice"]
    )

    data["LogCourseDuration"] = np.log1p(
        data["CourseDuration"]
    )

    data["RatingGap"] = (
        data["CourseRating"]
        -
        data["TeacherRating"]
    )

    data["AverageRating"] = (
        data["CourseRating"]
        +
        data["TeacherRating"]
    ) / 2

    data["CourseQualityScore"] = (
        data["CourseRating"]
        *
        data["TeacherRating"]
    )

    data["ExperienceRatingScore"] = (
        data["YearsOfExperience"]
        *
        data["TeacherRating"]
    )

    data["PriceRatingInteraction"] = (
        data["CoursePrice"]
        *
        data["AverageRating"]
    )

    data["PricePerRatingPoint"] = (
        data["CoursePrice"]
        /
        data["AverageRating"].replace(0, np.nan)
    )

    data["PricePerRatingPoint"] = (
        data["PricePerRatingPoint"]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )

    data["Category_Type"] = (
        data["CourseCategory"].astype(str)
        + "_"
        + data["CourseType"].astype(str)
    )

    data["Category_Level"] = (
        data["CourseCategory"].astype(str)
        + "_"
        + data["CourseLevel"].astype(str)
    )

    data["Type_Level"] = (
        data["CourseType"].astype(str)
        + "_"
        + data["CourseLevel"].astype(str)
    )

    data["Category_Expertise"] = (
        data["CourseCategory"].astype(str)
        + "_"
        + data["Expertise"].astype(str)
    )

    data["Level_Expertise"] = (
        data["CourseLevel"].astype(str)
        + "_"
        + data["Expertise"].astype(str)
    )

    data["ExperienceBand"] = pd.cut(
        data["YearsOfExperience"],
        bins=[
            -np.inf,
            5,
            10,
            20,
            np.inf
        ],
        labels=[
            "Early",
            "Developing",
            "Experienced",
            "Highly_Experienced"
        ]
    )

    return data


# ============================================================
# APPLICATION HEADER
# ============================================================

st.title("📊 EduPro Predictive Intelligence")

st.subheader(
    "Course Demand & Revenue Prediction"
)

st.write(
    """
    Use the final Day 14 machine-learning pipelines to
    estimate course enrollment demand and expected revenue.
    """
)

st.divider()


# ============================================================
# INPUT SECTION
# ============================================================

st.header("Course Information")

col1, col2 = st.columns(2)


with col1:

    course_category = st.selectbox(
        "Course Category",
        [
            "Technology",
            "Business",
            "Design",
            "Marketing",
            "Data Science"
        ]
    )

    course_type = st.selectbox(
        "Course Type",
        [
            "Online",
            "Offline",
            "Hybrid"
        ]
    )

    course_level = st.selectbox(
        "Course Level",
        [
            "Beginner",
            "Intermediate",
            "Advanced"
        ]
    )

    expertise = st.selectbox(
        "Teacher Expertise",
        [
            "Programming",
            "Data Science",
            "Business",
            "Design",
            "Marketing"
        ]
    )

    course_price = st.number_input(
        "Course Price",
        min_value=0.0,
        value=500.0,
        step=10.0
    )


with col2:

    course_duration = st.number_input(
        "Course Duration",
        min_value=0.1,
        value=7.5,
        step=0.5
    )

    course_rating = st.number_input(
        "Course Rating",
        min_value=0.0,
        max_value=5.0,
        value=4.5,
        step=0.01
    )

    teacher_rating = st.number_input(
        "Teacher Rating",
        min_value=0.0,
        max_value=5.0,
        value=4.5,
        step=0.01
    )

    years_experience = st.number_input(
        "Teacher Years of Experience",
        min_value=0.0,
        value=10.0,
        step=1.0
    )


# ============================================================
# PREDICTION
# ============================================================

st.divider()

predict_button = st.button(
    "🔮 Predict Course Demand & Revenue",
    type="primary",
    width="stretch"
)


if predict_button:

    input_data = pd.DataFrame([
        {
            "CourseCategory": course_category,
            "CourseType": course_type,
            "CourseLevel": course_level,
            "CoursePrice": course_price,
            "CourseDuration": course_duration,
            "CourseRating": course_rating,
            "TeacherRating": teacher_rating,
            "YearsOfExperience": years_experience,
            "Expertise": expertise
        }
    ])


    # --------------------------------------------------------
    # Revenue prediction
    # Original 9 features
    # --------------------------------------------------------

    revenue_prediction = (
        revenue_model.predict(
            input_data[
                [
                    "CourseCategory",
                    "CourseType",
                    "CourseLevel",
                    "CoursePrice",
                    "CourseDuration",
                    "CourseRating",
                    "TeacherRating",
                    "YearsOfExperience",
                    "Expertise"
                ]
            ]
        )[0]
    )


    # --------------------------------------------------------
    # Enrollment prediction
    # Engineered 26 features
    # --------------------------------------------------------

    engineered_data = (
        create_engineered_features(
            input_data
        )
    )


    enrollment_prediction = (
        enrollment_model.predict(
            engineered_data[
                [
                    "CourseCategory",
                    "CourseType",
                    "CourseLevel",
                    "CoursePrice",
                    "CourseDuration",
                    "CourseRating",
                    "TeacherRating",
                    "YearsOfExperience",
                    "Expertise",
                    "PricePerDay",
                    "PriceSquared",
                    "DurationSquared",
                    "LogCoursePrice",
                    "LogCourseDuration",
                    "RatingGap",
                    "AverageRating",
                    "CourseQualityScore",
                    "ExperienceRatingScore",
                    "PriceRatingInteraction",
                    "PricePerRatingPoint",
                    "Category_Type",
                    "Category_Level",
                    "Type_Level",
                    "Category_Expertise",
                    "Level_Expertise",
                    "ExperienceBand"
                ]
            ]
        )[0]
    )


    # --------------------------------------------------------
    # Business-safe output
    # --------------------------------------------------------

    enrollment_prediction = max(
        0,
        enrollment_prediction
    )

    revenue_prediction = max(
        0,
        revenue_prediction
    )


    # --------------------------------------------------------
    # Display predictions
    # --------------------------------------------------------

    st.success(
        "Prediction completed successfully."
    )

    result_col1, result_col2 = st.columns(2)


    with result_col1:

        st.metric(
            "Predicted Enrollment Count",
            f"{enrollment_prediction:.0f}"
        )


    with result_col2:

        st.metric(
            "Predicted Course Revenue",
            f"₹{revenue_prediction:,.2f}"
        )


    st.divider()

    st.subheader(
        "Prediction Input"
    )

    st.dataframe(
        input_data,
        use_container_width=True,
        hide_index=True
    )


# ============================================================
# MODEL INFORMATION
# ============================================================

with st.expander("Model Information"):

    st.write(
        """
        **EnrollmentCount**
        
        Random Forest using engineered features.
        
        **CourseRevenue**
        
        Linear Regression using original features.
        """
    )

    st.caption(
        "Models trained and finalized in Day 14."
    )
'''

with open(
    app_file_path,
    "w",
    encoding="utf-8"
) as f:

    f.write(app_code)


print("Streamlit application created:")
print(app_file_path)

Streamlit application created:
D:\Data Analytics Project\EduPro_Predictive_Modeling\app\app.py


## Create requriments.txt

In [12]:
# ============================================================
# CREATE REQUIREMENTS FILE
# ============================================================

requirements = """streamlit
pandas
numpy
scikit-learn
joblib
openpyxl
"""

with open(
    requirements_file_path,
    "w",
    encoding="utf-8"
) as f:

    f.write(requirements)

print(
    "Requirements file created:"
)

print(
    requirements_file_path
)

Requirements file created:
D:\Data Analytics Project\EduPro_Predictive_Modeling\app\requirements.txt


## Vlaidate genrated files

In [13]:
# ============================================================
# STREAMLIT FILE VALIDATION
# ============================================================

print("========== DAY 15 FILE VALIDATION ==========")

print("\napp.py:")
print(app_file_path)
print("Exists:", app_file_path.exists())

print("\nrequirements.txt:")
print(requirements_file_path)
print(
    "Exists:",
    requirements_file_path.exists()
)

if not app_file_path.exists():
    raise FileNotFoundError(
        "Streamlit app.py was not created."
    )

if not requirements_file_path.exists():
    raise FileNotFoundError(
        "requirements.txt was not created."
    )

print(
    "\nAll Day 15 application files created successfully."
)

========== DAY 15 FILE VALIDATION ==========

app.py:
D:\Data Analytics Project\EduPro_Predictive_Modeling\app\app.py
Exists: True

requirements.txt:
D:\Data Analytics Project\EduPro_Predictive_Modeling\app\requirements.txt
Exists: True

All Day 15 application files created successfully.


## Inspect app contents

In [14]:
# ============================================================
# VERIFY APP CONTENT
# ============================================================

with open(
    app_file_path,
    "r",
    encoding="utf-8"
) as f:

    app_text = f.read()

required_app_components = [
    "streamlit",
    "EduPro_Final_Enrollment_Model.joblib",
    "EduPro_Final_Revenue_Model.joblib",
    "create_engineered_features",
    "st.selectbox",
    "st.number_input",
    "st.button",
    "st.metric"
]

print(
    "========== APP CONTENT CHECK =========="
)

for component in required_app_components:

    found = component in app_text

    print(
        f"{component}: {found}"
    )

    if not found:
        raise AssertionError(
            f"Required application component missing: "
            f"{component}"
        )

print(
    "\nStreamlit application structure validated."
)

========== APP CONTENT CHECK ==========
streamlit: True
EduPro_Final_Enrollment_Model.joblib: True
EduPro_Final_Revenue_Model.joblib: True
create_engineered_features: True
st.selectbox: True
st.number_input: True
st.button: True
st.metric: True

Streamlit application structure validated.


## Final Day 15 validation

In [15]:
# ============================================================
# DAY 15 FINAL VALIDATION
# ============================================================

assert enrollment_model_path.exists()
assert revenue_model_path.exists()

assert app_file_path.exists()
assert requirements_file_path.exists()

assert len(original_features) == 9
assert len(engineered_features) == 17
assert len(all_modeling_features) == 26

assert (
    "create_engineered_features"
    in app_text
)

assert (
    "EduPro_Final_Enrollment_Model.joblib"
    in app_text
)

assert (
    "EduPro_Final_Revenue_Model.joblib"
    in app_text
)

print(
    "=============================================="
)

print(
    "DAY 15 STREAMLIT SETUP"
)

print(
    "VALIDATION PASSED"
)

print(
    "=============================================="
)

print("\nStreamlit app:")
print(app_file_path)

print("\nRequirements:")
print(requirements_file_path)

print("\nEnrollment model:")
print(enrollment_model_path)

print("\nRevenue model:")
print(revenue_model_path)

print(
    "\n=============================================="
)

DAY 15 STREAMLIT SETUP
VALIDATION PASSED

Streamlit app:
D:\Data Analytics Project\EduPro_Predictive_Modeling\app\app.py

Requirements:
D:\Data Analytics Project\EduPro_Predictive_Modeling\app\requirements.txt

Enrollment model:
D:\Data Analytics Project\EduPro_Predictive_Modeling\models\EduPro_Final_Enrollment_Model.joblib

Revenue model:
D:\Data Analytics Project\EduPro_Predictive_Modeling\models\EduPro_Final_Revenue_Model.joblib

